In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers peft accelerate huggingface_hub
!pip install -q sentence-transformers faiss-cpu
!pip install -q gradio Pillow
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.9 MB/s eta 0:00:00
✅ All packages installed!


In [2]:
# Cell 2 — Login to HuggingFace Hub
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in to HuggingFace Hub!')
except Exception:
    login()

✅ Logged in to HuggingFace Hub!


In [3]:
# Cell 3 — Load ViT emotion model
from transformers import ViTForImageClassification, AutoImageProcessor
import torch

MODEL_NAME = 'kashanikram/facial-emotion-vit'
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print('Loading emotion model...')
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = ViTForImageClassification.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f'✅ Model loaded!')
print(f'Device: {device}')

Loading emotion model...


preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/842 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Model loaded!
Device: cpu


In [4]:
# Cell 4 — Load RAG indexes
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
import faiss
import pickle

REPO_ID = 'kashanikram/facial-emotion-vit'
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('Loading RAG indexes...')
emotion_indexes = {}
emotion_chunks  = {}

# Download and load each emotion index
for emotion in EMOTIONS:
    index_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=f'faiss_{emotion}.bin',
        repo_type='model'
    )
    emotion_indexes[emotion] = faiss.read_index(index_path)

# Download chunks
chunks_path = hf_hub_download(
    repo_id=REPO_ID,
    filename='emotion_chunks.pkl',
    repo_type='model'
)
with open(chunks_path, 'rb') as f:
    emotion_chunks = pickle.load(f)

print('✅ All RAG indexes loaded!')
for emotion in EMOTIONS:
    print(f'  {emotion}: {emotion_indexes[emotion].ntotal} vectors')

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

faiss_neutral.bin:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

faiss_sad.bin:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

faiss_surprise.bin:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

emotion_chunks.pkl:   0%|          | 0.00/7.68k [00:00<?, ?B/s]

✅ All RAG indexes loaded!
  angry: 10 vectors
  disgust: 10 vectors
  fear: 10 vectors
  happy: 10 vectors
  neutral: 10 vectors
  sad: 10 vectors
  surprise: 10 vectors


In [5]:
# Cell 5 — Emotion detection + RAG + Agentic router
import torch
import numpy as np
from PIL import Image

def detect_emotion(image):
    """Image se emotion detect karo"""
    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)[0]
    top_idx = probs.argmax().item()
    emotion = model.config.id2label[top_idx]
    confidence = probs[top_idx].item()

    # Top 3 predictions
    top3 = sorted(enumerate(probs.tolist()), key=lambda x: x[1], reverse=True)[:3]
    top3_results = [(model.config.id2label[i], round(score*100, 1)) for i, score in top3]

    return emotion, round(confidence*100, 1), top3_results

def search_resources(emotion, top_k=3):
    """Emotion ke liye RAG resources fetch karo"""
    query = f"I am feeling {emotion} and need support"
    query_embedding = embedder.encode([query]).astype('float32')
    index = emotion_indexes[emotion]
    chunks = emotion_chunks[emotion]
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

def agent_router(emotion, confidence):
    """Agent decide karta hai kya response dena hai"""
    concerning = ['sad', 'angry', 'fear', 'disgust']
    positive   = ['happy']
    neutral    = ['neutral', 'surprise']

    if confidence < 40:
        return 'uncertain'
    elif emotion in concerning:
        return 'support'
    elif emotion in positive:
        return 'positive'
    else:
        return 'neutral'

def run_agent(image):
    """Complete pipeline — image se response tak"""
    if image is None:
        return "Please upload a face image."

    # Step 1: Detect emotion
    emotion, confidence, top3 = detect_emotion(image)

    # Step 2: Route
    route = agent_router(emotion, confidence)

    # Step 3: Build response
    response = f"Detected emotion: {emotion.upper()} ({confidence}%)\n\n"
    response += "Top predictions:\n"
    for e, c in top3:
        bar = '█' * int(c/5)
        response += f"  {e:10} {bar} {c}%\n"
    response += "\n"

    if route == 'uncertain':
        response += "Note: Low confidence detection. Please try with a clearer face image.\n\n"

    # Step 4: RAG resources
    resources = search_resources(emotion, top_k=3)

    if route == 'support':
        response += "We noticed you might be going through something difficult.\nHere are some resources that may help:\n\n"
    elif route == 'positive':
        response += "Great to see you are feeling good! Here are some tips to maintain wellbeing:\n\n"
    else:
        response += "Here are some mental wellness resources:\n\n"

    for i, resource in enumerate(resources, 1):
        response += f"{i}. {resource}\n\n"

    response += "---\nRemember: If you are in crisis, call or text 9-8-8 (Canada) — available 24/7."

    return response

print('✅ Agent pipeline ready!')

# Quick test
from PIL import Image
import numpy as np
test_img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
result = run_agent(test_img)
print('Test run successful!')

✅ Agent pipeline ready!
Test run successful!


In [6]:
# Cell 6 — Gradio UI
import gradio as gr

def analyze_emotion(image):
    if image is None:
        return "Please upload a face image."
    try:
        pil_image = Image.fromarray(image)
        return run_agent(pil_image)
    except Exception as e:
        return f"Error: {str(e)}\nPlease try with a clear face image."

demo = gr.Interface(
    fn=analyze_emotion,
    inputs=gr.Image(
        label="Upload a face image",
        type="numpy"
    ),
    outputs=gr.Textbox(
        label="Emotion Analysis + Mental Health Resources",
        lines=20
    ),
    title="Facial Emotion Recognition + Mental Health Navigator",
    description="Upload a face image — AI detects the emotion and provides relevant Canadian mental health resources. Powered by ViT + RAG + Agentic AI.",
    examples=[],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://90d40be619912815b6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
# Cell 7 — Summary before deployment
print('='*50)
print('NOTEBOOK 3 COMPLETE!')
print('='*50)
print('Model: kashanikram/facial-emotion-vit')
print('Accuracy: 65.74%')
print('RAG: 70 Canadian mental health resources')
print('Agent: 3 routes — support, positive, neutral')
print('UI: Gradio — working!')
print('='*50)
print('Next: Deploy to HuggingFace Spaces!')

NOTEBOOK 3 COMPLETE!
Model: kashanikram/facial-emotion-vit
Accuracy: 65.74%
RAG: 70 Canadian mental health resources
Agent: 3 routes — support, positive, neutral
UI: Gradio — working!
Next: Deploy to HuggingFace Spaces!
